In [5]:
pip install scanpy AnnData numpy pandas

  Using cached threadpoolctl-3.6.0-py3-none-any.whl.metadata (13 kB)
   ---------------------------------------- 0.0/2.2 MB ? eta -:--:--
   ---------------------------------------- 0.0/2.2 MB ? eta -:--:--
   --------- ------------------------------ 0.5/2.2 MB 2.6 MB/s eta 0:00:01
   ----------------------------- ---------- 1.6/2.2 MB 3.8 MB/s eta 0:00:01
   ---------------------------------------- 2.2/2.2 MB 4.2 MB/s  0:00:00
   ---------------------------------------- 0.0/10.0 MB ? eta -:--:--
   --- ------------------------------------ 0.8/10.0 MB 3.6 MB/s eta 0:00:03
   ------ --------------------------------- 1.6/10.0 MB 4.2 MB/s eta 0:00:02
   ---------- ----------------------------- 2.6/10.0 MB 4.4 MB/s eta 0:00:02
   --------------- ------------------------ 3.9/10.0 MB 4.9 MB/s eta 0:00:02
   ------------------ --------------------- 4.7/10.0 MB 5.0 MB/s eta 0:00:02
   -------------------- ------------------- 5.0/10.0 MB 4.8 MB/s eta 0:00:02
   -------------------- ------------

  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.

[notice] A new release of pip is available: 25.3 -> 26.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [7]:
pip install --upgrade pip

   ---------------------------------------- 0.0/1.8 MB ? eta -:--:--
   ---------------------------------------- 0.0/1.8 MB ? eta -:--:--
   ----- ---------------------------------- 0.3/1.8 MB ? eta -:--:--
   ----------------------- ---------------- 1.0/1.8 MB 3.9 MB/s eta 0:00:01
   ---------------------------------------- 1.8/1.8 MB 4.5 MB/s  0:00:00
  Attempting uninstall: pip
    Found existing installation: pip 25.3
    Uninstalling pip-25.3:
      Successfully uninstalled pip-25.3
Note: you may need to restart the kernel to use updated packages.


  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.


In [8]:
pip install --upgrade typing_extensions

Note: you may need to restart the kernel to use updated packages.


In [1]:
import scanpy as sc
import anndata as ad
import numpy as np
import pandas as pd

In [3]:
# 下载 PBMC 3k
adata = sc.datasets.pbmc3k()

# 保存为 h5ad 文件
adata.write_h5ad("../data_raw/pbmc3k_raw.h5ad")
print("✅ 已保存 pbmc3k_raw.h5ad")

✅ 已保存 pbmc3k_raw.h5ad


In [5]:
adata = ad.read_h5ad("../data_raw/pbmc3k_raw.h5ad")
print(adata)

AnnData object with n_obs × n_vars = 2700 × 32738
    var: 'gene_ids'
    layers: None (.X)


In [7]:
# 1. 整体概览 
print(adata)

# 2. 细胞级注释 (obs) 
print("\n【adata.obs】前 5 行：")
print(adata.obs.head())
print(f"\nobs 列名: {list(adata.obs.columns)}")
print(f"obs 形状: {adata.obs.shape}")

# 3. 基因级注释 (var) 
print("\n【adata.var】前 5 列：")
print(adata.var.head())
print(f"\nvar 列名: {list(adata.var.columns)}")

#  4. 多维数组注释 (obsm / varm) 
print(f"\n【adata.obsm】keys: {list(adata.obsm.keys())}")
print(f"【adata.varm】keys: {list(adata.varm.keys())}")

#  5. 非结构化注释 (uns) 
print(f"\n【adata.uns】keys: {list(adata.uns.keys())}")

#  6. 主矩阵 X 的基本统计 
print(f"\n【adata.X】")
print(f"  类型: {type(adata.X)}")
print(f"  形状: {adata.X.shape}")
print(f"  数据类型: {adata.X.dtype}")
print(f"  稀疏度: {1 - adata.X.nnz / np.prod(adata.X.shape):.2%}")

AnnData object with n_obs × n_vars = 2700 × 32738
    var: 'gene_ids'
    layers: None (.X)

【adata.obs】前 5 行：
Empty DataFrame
Columns: []
Index: [AAACATACAACCAC-1, AAACATTGAGCTAC-1, AAACATTGATCAGC-1, AAACCGTGCTTCCG-1, AAACCGTGTATGCG-1]

obs 列名: []
obs 形状: (2700, 0)

【adata.var】前 5 列：
                     gene_ids
index                        
MIR1302-10    ENSG00000243485
FAM138A       ENSG00000237613
OR4F5         ENSG00000186092
RP11-34P13.7  ENSG00000238009
RP11-34P13.8  ENSG00000239945

var 列名: ['gene_ids']

【adata.obsm】keys: []
【adata.varm】keys: []

【adata.uns】keys: []

【adata.X】
  类型: <class 'scipy.sparse._csr.csr_matrix'>
  形状: (2700, 32738)
  数据类型: float32
  稀疏度: 97.41%


In [8]:
adata.var['mt'] = adata.var_names.str.startswith('MT-')
print(f"线粒体基因数量: {adata.var['mt'].sum()}")
# 质控指标
sc.pp.calculate_qc_metrics(adata, qc_vars=['mt'], percent_top=None, log1p=False, inplace=True)
print("添加 QC 指标后的 obs 列：")
print(list(adata.obs.columns))
# ['n_genes_by_counts', 'total_counts', 'total_counts_mt', 'pct_counts_mt']

#过滤
sc.pp.filter_cells(adata, min_genes=200)        # 至少检测到 200 个基因
sc.pp.filter_genes(adata, min_cells=3)          # 基因至少在 3 个细胞中表达
adata = adata[adata.obs['pct_counts_mt'] < 20].copy()  # 线粒体比例 < 20%

print(f"筛选后: {adata.n_obs} 个细胞, {adata.n_vars} 个基因")

# 标准化 + PCA，产生 obsm
sc.pp.normalize_total(adata, target_sum=1e4)
sc.pp.log1p(adata)
adata.raw = adata
sc.pp.highly_variable_genes(adata, n_top_genes=2000)
sc.tl.pca(adata, n_comps=50)

print(f"\n处理后 obsm keys: {list(adata.obsm.keys())}")
# ['X_pca']
print(f"X_pca 形状: {adata.obsm['X_pca'].shape}")
# (2700, 50)

线粒体基因数量: 13
添加 QC 指标后的 obs 列：
['n_genes_by_counts', 'total_counts', 'total_counts_mt', 'pct_counts_mt']
筛选后: 2698 个细胞, 13714 个基因

处理后 obsm keys: ['X_pca']
X_pca 形状: (2698, 50)


In [9]:
# ---------- 按条件筛选（如 total_counts 最高的 100 个）----------
top100_idx = adata.obs['total_counts'].nlargest(100).index
adata_100 = adata[top100_idx].copy()

print(f"筛选后: {adata_100}")
print(f"  n_obs = {adata_100.n_obs}")   # 100
print(f"  n_vars = {adata_100.n_vars}") # 32738（基因数不变）
print(f"  obsm keys = {list(adata_100.obsm.keys())}")  # X_pca 自动跟随
print(f"  X_pca shape = {adata_100.obsm['X_pca'].shape}")  # (100, 50) ✅

筛选后: AnnData object with n_obs × n_vars = 100 × 13714
    obs: 'n_genes_by_counts', 'total_counts', 'total_counts_mt', 'pct_counts_mt', 'n_genes'
    var: 'gene_ids', 'mt', 'n_cells_by_counts', 'mean_counts', 'pct_dropout_by_counts', 'total_counts', 'n_cells', 'highly_variable', 'means', 'dispersions', 'dispersions_norm'
    uns: 'log1p', 'hvg', 'pca'
    obsm: 'X_pca'
    varm: 'PCs'
    layers: None (.X)
  n_obs = 100
  n_vars = 13714
  obsm keys = ['X_pca']
  X_pca shape = (100, 50)


In [11]:
output_path = "../data_processed/pbmc3k_100cells.h5ad"
adata_100.write_h5ad(output_path)
print(f"✅ 已保存: {output_path}")

# 验证：重新读取
adata_check = ad.read_h5ad(output_path)
print(f"验证读取: {adata_check}")
# AnnData object with n_obs × n_vars = 100 × 32738

✅ 已保存: ../data_processed/pbmc3k_100cells.h5ad
验证读取: AnnData object with n_obs × n_vars = 100 × 13714
    obs: 'n_genes_by_counts', 'total_counts', 'total_counts_mt', 'pct_counts_mt', 'n_genes'
    var: 'gene_ids', 'mt', 'n_cells_by_counts', 'mean_counts', 'pct_dropout_by_counts', 'total_counts', 'n_cells', 'highly_variable', 'means', 'dispersions', 'dispersions_norm'
    uns: 'hvg', 'log1p', 'pca'
    obsm: 'X_pca'
    varm: 'PCs'
    layers: None (.X)
